In [41]:
# importing all the neccesary libraires 
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import scipy.stats as stats
from scipy.stats import skew
from scipy.stats import zscore
%matplotlib inline

In [ ]:
# Loading the training dataset
def load_data_in_chunks(file_path, chunk_size):
    for chunk in pd.read_csv(file_path, chunksize=chunk_size):
        yield chunk  

# To load the data into a single variable
podcast_data = pd.concat(load_data_in_chunks('train.csv', 10000), ignore_index=True)
podcast_data.head(5)

,id,Podcast_Name,Episode_Title,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes
0,0,Mystery Matters,Episode 98,NaN,True Crime,74.81,Thursday,Night,NaN,0.0,Positive,31.41998
1,1,Joke Junction,Episode 26,119.80,Comedy,66.95,Saturday,Afternoon,75.95,2.0,Negative,88.01241
2,2,Study Sessions,Episode 16,73.90,Education,69.97,Tuesday,Evening,8.97,0.0,Negative,44.92531
3,3,Digital Digest,Episode 45,67.17,Technology,57.22,Monday,Morning,78.70,2.0,Positive,46.27824
4,4,Mind & Body,Episode 86,110.51,Health,80.07,Monday,Afternoon,58.68,3.0,Neutral,75.61031


In [4]:
#Data Shape
podcast_data.shape

(750000, 12)

In [5]:
# Data Types
podcast_data.dtypes

id                               int64
Podcast_Name                    object
Episode_Title                   object
Episode_Length_minutes         float64
Genre                           object
Host_Popularity_percentage     float64
Publication_Day                 object
Publication_Time                object
Guest_Popularity_percentage    float64
Number_of_Ads                  float64
Episode_Sentiment               object
Listening_Time_minutes         float64
dtype: object

In [ ]:
# Null values 
missing = podcast_data.isnull().sum()
missing

id                                  0
Podcast_Name                        0
Episode_Title                       0
Episode_Length_minutes          87093
Genre                               0
Host_Popularity_percentage          0
Publication_Day                     0
Publication_Time                    0
Guest_Popularity_percentage    146030
Number_of_Ads                       1
Episode_Sentiment                   0
Listening_Time_minutes              0
dtype: int64

In [ ]:
# percentage of Null values 
percentage_missing = (missing / len(podcast_data)) * 100
percentage_missing.sort_values(ascending=False)

Guest_Popularity_percentage    19.470667
Episode_Length_minutes         11.612400
Number_of_Ads                   0.000133
id                              0.000000
Episode_Title                   0.000000
Podcast_Name                    0.000000
Genre                           0.000000
Host_Popularity_percentage      0.000000
Publication_Time                0.000000
Publication_Day                 0.000000
Episode_Sentiment               0.000000
Listening_Time_minutes          0.000000
dtype: float64

In [19]:
# Target variable 
podcast_data['Listening_Time_minutes'].notnull().sum()

np.int64(750000)

In [40]:
# Checking the min, max, mean of the dataset
podcast_data.describe(include='all')

,id,Podcast_Name,Episode_Title,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes
count,750000.000000,750000,750000,662907.000000,750000,750000.000000,750000,750000,603970.000000,749999.000000,750000,750000.000000
unique,NaN,48,100,NaN,10,NaN,7,4,NaN,NaN,3,NaN
top,NaN,Tech Talks,Episode 71,NaN,Sports,NaN,Sunday,Night,NaN,NaN,Neutral,NaN
freq,NaN,22847,10515,NaN,87606,NaN,115946,196849,NaN,NaN,251291,NaN
mean,374999.500000,NaN,NaN,64.504738,NaN,59.859901,NaN,NaN,52.236449,1.348855,NaN,45.437406
std,216506.495284,NaN,NaN,32.969603,NaN,22.873098,NaN,NaN,28.451241,1.151130,NaN,27.138306
min,0.000000,NaN,NaN,0.000000,NaN,1.300000,NaN,NaN,0.000000,0.000000,NaN,0.000000
25%,187499.750000,NaN,NaN,35.730000,NaN,39.410000,NaN,NaN,28.380000,0.000000,NaN,23.178350
50%,374999.500000,NaN,NaN,63.840000,NaN,60.050000,NaN,NaN,53.580000,1.000000,NaN,43.379460
75%,562499.250000,NaN,NaN,94.070000,NaN,79.530000,NaN,NaN,76.600000,2.000000,NaN,64.811580


In [22]:
# checking skweness 
stats.skew(podcast_data['Listening_Time_minutes'])

np.float64(0.350811556885081)

In [ ]:
# Checking Kurtosis
stats.kurtosis(podcast_data['Listening_Time_minutes'])

np.float64(-0.6612398847652892)

In [32]:
# Summary of the numerical columns and correlation to the target variable
target = 'Listening_Time_minutes'
numerical_cols = podcast_data.select_dtypes(include=[np.number]).columns.drop(target)

stats = []

for col in numerical_cols:
    count_non_null = podcast_data[col].count()
    mean = podcast_data[col].mean()
    median = podcast_data[col].median()
    min_val = podcast_data[col].min()
    max_val = podcast_data[col].max()
    std_dev = podcast_data[col].std()
    missing_values = podcast_data[col].isnull().sum()
    skewness = skew(podcast_data[col].dropna())
    correlation = podcast_data[[col, target]].corr().iloc[0, 1]
    
    stats.append({
        'Column': col,
        'Count': count_non_null,
        'Mean': mean,
        'Median': median,
        'Min': min_val,
        'Max': max_val,
        'Std Dev': std_dev,
        'Missing Values': missing_values,
        'Skewness': skewness,
        'Correlation w/ Target': correlation
    })

summary_df = pd.DataFrame(stats)
summary_df

,Column,Count,Mean,Median,Min,Max,Std Dev,Missing Values,Skewness,Correlation w/ Target
0,id,750000,374999.500000,374999.50,0.0,749999.00,216506.495284,0,-1.963096e-15,-0.000876
1,Episode_Length_minutes,662907,64.504738,63.84,0.0,325.24,32.969603,87093,-2.005608e-03,0.916749
2,Host_Popularity_percentage,750000,59.859901,60.05,1.3,119.46,22.873098,0,4.926265e-03,0.050870
3,Guest_Popularity_percentage,603970,52.236449,53.58,0.0,119.91,28.451241,146030,-1.070351e-01,-0.016014
4,Number_of_Ads,749999,1.348855,1.00,0.0,103.91,1.151130,1,6.032980e+00,-0.118337


In [37]:
# List of potential ID columns
id_cols = ['id', 'Episode_Title', 'Podcast_Name']  # adjust as needed

for col in id_cols:
    print(f"Column: {col}")
    print("Is Unique?:", podcast_data[col].is_unique)
    print("Number of Unique Values:", podcast_data[col].nunique())
    print()


Column: id
Is Unique?: True
Number of Unique Values: 750000

Column: Episode_Title
Is Unique?: False
Number of Unique Values: 100

Column: Podcast_Name
Is Unique?: False
Number of Unique Values: 48



In [38]:
# Publication_Day
print("Unique Publication Days:", podcast_data['Publication_Day'].unique())
print("\nPublication Day Counts:\n", podcast_data['Publication_Day'].value_counts())

# Publication_Time
print("\nUnique Publication Times:", podcast_data['Publication_Time'].unique())
print("\nPublication Time Counts:\n", podcast_data['Publication_Time'].value_counts())


Unique Publication Days: ['Thursday' 'Saturday' 'Tuesday' 'Monday' 'Sunday' 'Wednesday' 'Friday']

Publication Day Counts:
 Publication_Day
Sunday       115946
Monday       111963
Friday       108237
Wednesday    107886
Thursday     104360
Saturday     103505
Tuesday       98103
Name: count, dtype: int64

Unique Publication Times: ['Night' 'Afternoon' 'Evening' 'Morning']

Publication Time Counts:
 Publication_Time
Night        196849
Evening      195778
Afternoon    179460
Morning      177913
Name: count, dtype: int64


In [39]:
# 1. Missing percentage for each column
missing_percent = podcast_data.isnull().mean() * 100
print("Missing % per column:\n", missing_percent)

# 2. Columns with >20% missing
print("\nColumns with >20% missing:\n", missing_percent[missing_percent > 20])

# 3. Columns with >50% missing
print("\nColumns with >50% missing:\n", missing_percent[missing_percent > 50])

# 4. Check if rows missing Guest_Popularity_percentage also miss Episode_Length_minutes
both_missing = podcast_data[
    podcast_data['Guest_Popularity_percentage'].isnull() & 
    podcast_data['Episode_Length_minutes'].isnull()
]
print("\nRows missing both Guest_Popularity_percentage and Episode_Length_minutes:", len(both_missing))

# 5. Count rows with any missing value
rows_with_missing = podcast_data.isnull().any(axis=1).sum()
print("\nTotal rows with any missing value:", rows_with_missing)


Missing % per column:
 id                              0.000000
Podcast_Name                    0.000000
Episode_Title                   0.000000
Episode_Length_minutes         11.612400
Genre                           0.000000
Host_Popularity_percentage      0.000000
Publication_Day                 0.000000
Publication_Time                0.000000
Guest_Popularity_percentage    19.470667
Number_of_Ads                   0.000133
Episode_Sentiment               0.000000
Listening_Time_minutes          0.000000
dtype: float64

Columns with >20% missing:
 Series([], dtype: float64)

Columns with >50% missing:
 Series([], dtype: float64)

Rows missing both Guest_Popularity_percentage and Episode_Length_minutes: 22172

Total rows with any missing value: 210952


In [44]:
# Choose columns with missing values
cols_with_missing = ['Episode_Length_minutes', 'Guest_Popularity_percentage', 'Number_of_Ads']

# Create a correlation matrix of missing flags
missing_corr = podcast_data[cols_with_missing].isnull().astype(int).corr()

print("Missing Value Co-occurrence (correlation between missingness):")
print(missing_corr)


Missing Value Co-occurrence (correlation between missingness):
                             Episode_Length_minutes  \
Episode_Length_minutes                     1.000000   
Guest_Popularity_percentage                0.054805   
Number_of_Ads                             -0.000419   

                             Guest_Popularity_percentage  Number_of_Ads  
Episode_Length_minutes                          0.054805      -0.000419  
Guest_Popularity_percentage                     1.000000      -0.000568  
Number_of_Ads                                  -0.000568       1.000000  


In [43]:
# List of categorical columns
categorical_cols = ['Podcast_Name', 'Episode_Title', 'Genre',
                    'Publication_Day', 'Publication_Time', 'Episode_Sentiment']

# Print top 5 frequent values for each
for col in categorical_cols:
    print(f"\nTop 5 values for {col}:")
    print(podcast_data[col].value_counts().head(5))



Top 5 values for Podcast_Name:
Podcast_Name
Tech Talks       22847
Sports Weekly    20053
Funny Folks      19635
Tech Trends      19549
Fitness First    19488
Name: count, dtype: int64

Top 5 values for Episode_Title:
Episode_Title
Episode 71    10515
Episode 62    10373
Episode 31    10292
Episode 61     9991
Episode 69     9864
Name: count, dtype: int64

Top 5 values for Genre:
Genre
Sports        87606
Technology    86256
True Crime    85059
Lifestyle     82461
Comedy        81453
Name: count, dtype: int64

Top 5 values for Publication_Day:
Publication_Day
Sunday       115946
Monday       111963
Friday       108237
Wednesday    107886
Thursday     104360
Name: count, dtype: int64

Top 5 values for Publication_Time:
Publication_Time
Night        196849
Evening      195778
Afternoon    179460
Morning      177913
Name: count, dtype: int64

Top 5 values for Episode_Sentiment:
Episode_Sentiment
Neutral     251291
Negative    250116
Positive    248593
Name: count, dtype: int64


In [46]:
# List of numerical columns
numerical_cols = ['Episode_Length_minutes', 'Host_Popularity_percentage',
                  'Guest_Popularity_percentage', 'Number_of_Ads', 'Listening_Time_minutes']

# Dictionary to store outlier counts
z_outlier_counts = {}

# Loop through each column and compute Z-scores and outlier count
for col in numerical_cols:
    col_data = podcast_data[col].dropna()
    z = zscore(col_data)
    outliers = np.sum(np.abs(z) > 3)
    z_outlier_counts[col] = outliers

# Display result
z_outlier_counts

{'Episode_Length_minutes': np.int64(1),
 'Host_Popularity_percentage': np.int64(0),
 'Guest_Popularity_percentage': np.int64(0),
 'Number_of_Ads': np.int64(9),
 'Listening_Time_minutes': np.int64(0)}

# EDA FINDINGS

# 📊 Data Overview Report

## 🔹 Dataset Shape
- The dataset contains **750,000 rows** and **12 columns**.

---

## 🔹 Data Types

| Column Name                    | Data Type |
|-------------------------------|-----------|
| id                            | int64     |
| Podcast_Name                  | object    |
| Episode_Title                 | object    |
| Episode_Length_minutes        | float64   |
| Genre                         | object    |
| Host_Popularity_percentage    | float64   |
| Publication_Day               | object    |
| Publication_Time              | object    |
| Guest_Popularity_percentage   | float64   |
| Number_of_Ads                 | float64   |
| Episode_Sentiment             | object    |
| Listening_Time_minutes (Target)| float64  |

---

## 🔹 Missing Values

### 🔸 Count of Null Values

| Column                         | Null Count |
|--------------------------------|------------|
| Episode_Length_minutes         | 87,093     |
| Guest_Popularity_percentage    | 146,030    |
| Number_of_Ads                  | 1          |
| All Other Columns              | 0          |

### 🔸 Percentage of Null Values

| Column                         | % Missing |
|--------------------------------|-----------|
| Guest_Popularity_percentage    | 19.47%    |
| Episode_Length_minutes         | 11.61%    |
| Number_of_Ads                  | 0.00013%  |

- **Total rows with any missing value**: 210,952  
- **Rows missing both `Guest_Popularity_percentage` and `Episode_Length_minutes`**: 22,172  
- **No columns have >20% or >50% missing values**

---

## 🔹 Target Variable: `Listening_Time_minutes`

- **Data type**: float64  
- **Non-null count**: 750,000  
- **Skewness**: 0.35 (slightly right-skewed)  
- **Kurtosis**: -0.66 (slightly flatter than normal distribution)

---

## 🔹 Descriptive Statistics

### Full Dataset Summary (including categorical columns):

- `Podcast_Name`: 48 unique values (most frequent: *Tech Talks*, 22,847 occurrences)  
- `Episode_Title`: 100 unique values (most frequent: *Episode 71*, 10,515 occurrences)  
- `Genre`: 10 unique values (most frequent: *Sports*, 87,606 occurrences)  
- `Publication_Day`: 7 unique values (most frequent: *Sunday*, 115,946 occurrences)  
- `Publication_Time`: 4 unique values (most frequent: *Night*, 196,849 occurrences)  
- `Episode_Sentiment`: 3 unique values (most frequent: *Neutral*, 251,291 occurrences)

---

## 🔹 Summary of Numerical Columns

| Column                       | Count   | Mean     | Median   | Min   | Max    | Std Dev | Missing | Skewness | Corr. w/ Target |
|-----------------------------|---------|----------|----------|-------|--------|----------|---------|-----------|-----------------|
| id                          | 750,000 | 375000.0 | 375000.0 | 0.0   | 749999 | 216506.5 | 0       | ~0        | -0.0009         |
| Episode_Length_minutes      | 662,907 | 64.50    | 63.84    | 0.0   | 325.24 | 32.97    | 87,093  | -0.002    | 0.9167          |
| Host_Popularity_percentage  | 750,000 | 59.86    | 60.05    | 1.3   | 119.46 | 22.87    | 0       | 0.0049    | 0.0509          |
| Guest_Popularity_percentage | 603,970 | 52.24    | 53.58    | 0.0   | 119.91 | 28.45    | 146,030 | -0.107    | -0.0160         |
| Number_of_Ads               | 749,999 | 1.35     | 1.0      | 0.0   | 103.91 | 1.15     | 1       | 6.03      | -0.1183         |

---

## 🔹 Identifier Column Check

| Column         | Is Unique? | # of Unique Values |
|----------------|------------|---------------------|
| id             | ✅ Yes      | 750,000             |
| Episode_Title  | ❌ No       | 100                 |
| Podcast_Name   | ❌ No       | 48                  |

- Only the `id` column is a true identifier.

---

## 🔹 Date/Time Features

### Publication Day
- **Unique values**: 7 (`Sunday`, `Monday`, `Tuesday`, `Wednesday`, `Thursday`, `Friday`, `Saturday`)
- **Most frequent**: Sunday (115,946)

### Publication Time
- **Unique values**: 4 (`Night`, `Afternoon`, `Evening`, `Morning`)
- **Most frequent**: Night (196,849)

---

## 🔍 Minor Enhancements (Optional Insights)

### 🧮 Z-Score Based Outlier Count (Threshold: |z| > 3)

| Column                      | # of Outliers |
|----------------------------|---------------|
| Episode_Length_minutes     | 1             |
| Host_Popularity_percentage | 0             |
| Guest_Popularity_percentage| 0             |
| Number_of_Ads              | 9             |
| Listening_Time_minutes     | 0             |

---

### 🔤 Top 5 Frequent Values per Categorical Column

**Podcast_Name**  
- Tech Talks: 22,847  
- Sports Weekly: 20,053  
- Funny Folks: 19,635  
- Tech Trends: 19,549  
- Fitness First: 19,488  

**Episode_Title**  
- Episode 71: 10,515  
- Episode 62: 10,373  
- Episode 31: 10,292  
- Episode 61: 9,991  
- Episode 69: 9,864  

**Genre**  
- Sports: 87,606  
- Technology: 86,256  
- True Crime: 85,059  
- Lifestyle: 82,461  
- Comedy: 81,453  

**Publication_Day**  
- Sunday: 115,946  
- Monday: 111,963  
- Friday: 108,237  
- Wednesday: 107,886  
- Thursday: 104,360  

**Publication_Time**  
- Night: 196,849  
- Evening: 195,778  
- Afternoon: 179,460  
- Morning: 177,913  

**Episode_Sentiment**  
- Neutral: 251,291  
- Negative: 250,116  
- Positive: 248,593  

---

### 🔄 Missing Value Co-occurrence (Correlation of Missing Flags)

|                             | Episode_Length_minutes | Guest_Popularity_percentage | Number_of_Ads |
|-----------------------------|------------------------|-----------------------------|----------------|
| **Episode_Length_minutes**  | 1.000                  | 0.055                       | -0.0004        |
| **Guest_Popularity_percentage** | 0.055             | 1.000                       | -0.0006        |
| **Number_of_Ads**           | -0.0004                | -0.0006                     | 1.000          |